# 🚀 DRIFTFormer — A100 Training Notebook
**NAVDRIFT-0 · ISRO SIH 2026 · PS #26168**

Trains the DRIFTFormer Transformer on the IO-VNBD dataset to predict:
- `dx, dy` — position delta over a 64-step IMU window
- `dθ` — heading change (yaw delta)

Compatible with the NAVDRIFT-0 dashboard's DRIFTFormer inference block.


## 1 · Environment Setup

In [ ]:
# Verify GPU
!nvidia-smi
import torch
print(f'PyTorch {torch.__version__} | CUDA {torch.version.cuda} | GPU: {torch.cuda.get_device_name(0)}')
assert torch.cuda.is_available(), 'No GPU found — switch runtime to A100'
DEVICE = torch.device('cuda')
print('✅ Ready on', DEVICE)

In [ ]:
!pip install -q numpy pandas scikit-learn matplotlib tqdm einops
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import StandardScaler
import json, os, math, random
print('✅ Imports OK')

## 2 · Download IO-VNBD Dataset

In [ ]:
# Clone the IO-VNBD repo (contains synchronised vehicle + smartphone CSV files)
!git clone --depth 1 https://github.com/onyekpeu/IO-VNBD.git IO-VNBD
!find IO-VNBD -name '*.csv' | head -20
!find IO-VNBD -name '*.zip' | head -10

In [ ]:
import zipfile, glob

# Unzip any archives found
for zf in glob.glob('IO-VNBD/**/*.zip', recursive=True):
    out = Path(zf).parent / Path(zf).stem
    out.mkdir(exist_ok=True)
    print(f'Unzipping {zf} → {out}')
    with zipfile.ZipFile(zf) as z:
        z.extractall(out)

# Find all CSVs
csv_files = glob.glob('IO-VNBD/**/*.csv', recursive=True)
print(f'\n{len(csv_files)} CSV files found:')
for f in csv_files[:20]: print(' ', f)

In [ ]:
# Preview one file to understand column structure
if csv_files:
    sample = pd.read_csv(csv_files[0])
    print('Shape:', sample.shape)
    print('Columns:', sample.columns.tolist())
    display(sample.head(5))
else:
    print('No CSVs found — check repo structure above')

## 3 · Column Mapping
Edit the dict below to match your actual column names from the preview above.

In [ ]:
# ── EDIT THESE to match your actual column names ──────────────────────────
COL = dict(
    time   = 'timestamp',      # seconds or ms
    ax     = 'accel_x',        # m/s² linear accel
    ay     = 'accel_y',
    az     = 'accel_z',
    gx     = 'gyro_x',         # rad/s angular rate
    gy     = 'gyro_y',
    gz     = 'gyro_z',
    lat    = 'latitude',        # GPS latitude  (degrees)
    lon    = 'longitude',       # GPS longitude (degrees)
    speed  = 'speed',           # m/s ground speed (set None if absent)
)
# ──────────────────────────────────────────────────────────────────────────

# Auto-detect columns if names don't match
def auto_map(df, col_map):
    cols = {c.lower(): c for c in df.columns}
    mapped = {}
    candidates = {
        'time':  ['timestamp','time','t','epoch'],
        'ax':    ['accel_x','acc_x','ax','linear_acceleration_x','a_x'],
        'ay':    ['accel_y','acc_y','ay','linear_acceleration_y','a_y'],
        'az':    ['accel_z','acc_z','az','linear_acceleration_z','a_z'],
        'gx':    ['gyro_x','gyr_x','gx','angular_velocity_x','g_x'],
        'gy':    ['gyro_y','gyr_y','gy','angular_velocity_y','g_y'],
        'gz':    ['gyro_z','gyr_z','gz','angular_velocity_z','g_z'],
        'lat':   ['latitude','lat','gps_lat'],
        'lon':   ['longitude','lon','lng','gps_lon'],
        'speed': ['speed','velocity','spd','gps_speed'],
    }
    for key, cands in candidates.items():
        for c in cands:
            if c in cols:
                mapped[key] = cols[c]
                break
        if key not in mapped:
            mapped[key] = col_map.get(key)  # fall back to user setting
    return mapped

if csv_files:
    sample = pd.read_csv(csv_files[0])
    COL = auto_map(sample, COL)
    print('Column mapping:', COL)

## 4 · Dataset Builder

In [ ]:
SEQ_LEN  = 64     # input window (steps) — matches dashboard DRIFTFormer
STRIDE   = 8      # window stride for training samples
IMU_COLS = ['ax','ay','az','gx','gy','gz']  # 6 IMU channels

R_EARTH  = 6_371_000  # metres

def lat_lon_to_xy(lat_arr, lon_arr):
    """Convert lat/lon sequence to local x/y in metres (origin = first point)."""
    lat0, lon0 = lat_arr[0], lon_arr[0]
    x = np.deg2rad(lon_arr - lon0) * R_EARTH * np.cos(np.deg2rad(lat0))
    y = np.deg2rad(lat_arr - lat0) * R_EARTH
    return x, y

def load_drive(path, col):
    """Load one CSV drive, return (N,6) IMU array and (N,3) target array [dx,dy,dth]."""
    df = pd.read_csv(path)
    # rename columns
    rename = {v: k for k, v in col.items() if v and v in df.columns}
    df.rename(columns=rename, inplace=True)

    needed_imu = [c for c in ['ax','ay','az','gx','gy','gz'] if c in df.columns]
    if len(needed_imu) < 4:
        return None, None  # skip files without IMU

    # Fill missing IMU channels with zeros
    for c in ['ax','ay','az','gx','gy','gz']:
        if c not in df.columns: df[c] = 0.0

    imu = df[['ax','ay','az','gx','gy','gz']].values.astype(np.float32)

    # Build targets from GPS if available
    if 'lat' in df.columns and 'lon' in df.columns:
        x, y = lat_lon_to_xy(df['lat'].values, df['lon'].values)
        dx = np.diff(x, prepend=x[0]).astype(np.float32)
        dy = np.diff(y, prepend=y[0]).astype(np.float32)
        heading  = np.arctan2(dy, dx).astype(np.float32)
        dth = np.diff(heading, prepend=heading[0]).astype(np.float32)
        dth = np.clip(dth, -np.pi, np.pi)
    else:
        # Fall back: use gz integral as heading proxy
        dt = 0.1  # assume 10 Hz
        dth = (df['gz'].values * dt).astype(np.float32)
        dx = dy = np.zeros(len(imu), dtype=np.float32)

    targets = np.stack([dx, dy, dth], axis=1)  # (N, 3)
    return imu, targets


class IOVNBDDataset(Dataset):
    def __init__(self, csv_files, col_map, seq_len=64, stride=8, scaler=None, fit_scaler=True):
        self.seq_len = seq_len
        seqs_X, seqs_Y = [], []

        raw_X, raw_Y = [], []
        for f in tqdm(csv_files, desc='Loading drives'):
            imu, tgt = load_drive(f, col_map)
            if imu is None or len(imu) < seq_len + 1: continue
            raw_X.append(imu)
            raw_Y.append(tgt)

        # Fit scaler on ALL imu data
        all_imu = np.concatenate(raw_X, axis=0)
        if scaler is None:
            scaler = StandardScaler()
            scaler.fit(all_imu)
        self.scaler = scaler

        for imu, tgt in zip(raw_X, raw_Y):
            imu_n = scaler.transform(imu)
            N = len(imu_n)
            for i in range(0, N - seq_len, stride):
                seqs_X.append(imu_n[i:i+seq_len])    # (64, 6)
                # target = cumulative displacement over window
                window_tgt = tgt[i:i+seq_len]
                seqs_Y.append(window_tgt.sum(axis=0)) # (3,): [Δx, Δy, Δθ]

        self.X = torch.from_numpy(np.array(seqs_X, dtype=np.float32))  # (N,64,6)
        self.Y = torch.from_numpy(np.array(seqs_Y, dtype=np.float32))  # (N,3)
        print(f'Dataset: {len(self.X)} windows | X={self.X.shape} | Y={self.Y.shape}')

    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.Y[i]

print('✅ Dataset class defined')

In [ ]:
# Build dataset
dataset = IOVNBDDataset(csv_files, COL, seq_len=SEQ_LEN, stride=STRIDE)

# 80/10/10 split
n = len(dataset)
n_train = int(0.8 * n)
n_val   = int(0.1 * n)
n_test  = n - n_train - n_val
train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42))

BATCH = 512  # A100 can handle large batches
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=4, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=4, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=4, pin_memory=True)
print(f'Train: {n_train} | Val: {n_val} | Test: {n_test}')

## 5 · DRIFTFormer Model
12-head causal Transformer — matches the attention heatmap in the NAVDRIFT-0 dashboard.

In [ ]:
class DRIFTFormer(nn.Module):
    """
    Causal Transformer for IMU dead-reckoning.
    Input:  (B, SEQ_LEN, 6)  — normalised [ax,ay,az,gx,gy,gz]
    Output: (B, 3)           — [Δx, Δy, Δθ] over the window
    """
    def __init__(self, d_model=128, n_heads=12, n_layers=4,
                 d_ff=512, dropout=0.1, seq_len=64, n_imu=6, n_out=3):
        super().__init__()
        self.d_model = d_model

        # Input projection
        self.input_proj = nn.Linear(n_imu, d_model)

        # Learned positional embedding
        self.pos_emb = nn.Embedding(seq_len, d_model)

        # Causal Transformer encoder
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

        # Causal mask (upper triangular)
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
        self.register_buffer('causal_mask', mask)

        # Output head — pool last token + mean pool, then project
        self.head = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, n_out),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):
        B, T, _ = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0)  # (1, T)
        h = self.input_proj(x) + self.pos_emb(pos)           # (B, T, d)
        h = self.transformer(h, mask=self.causal_mask[:T,:T])
        # Pool: last token (most recent) + mean of all
        pooled = torch.cat([h[:, -1, :], h.mean(dim=1)], dim=-1)  # (B, 2*d)
        return self.head(pooled)  # (B, 3)


model = DRIFTFormer(
    d_model=128, n_heads=12, n_layers=4,
    d_ff=512, dropout=0.1, seq_len=SEQ_LEN
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'DRIFTFormer — {n_params/1e6:.2f}M parameters')
print(model)

## 6 · Training Loop

In [ ]:
EPOCHS   = 60
LR       = 3e-4
WD       = 1e-4
PATIENCE = 10      # early stopping patience
GRAD_CLIP= 1.0
SAVE_PATH= 'driftformer_best.pt'

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# Loss: weighted MSE — emphasise heading more than position
# [dx_weight, dy_weight, dtheta_weight]
LOSS_WEIGHTS = torch.tensor([1.0, 1.0, 5.0], device=DEVICE)

def weighted_mse(pred, tgt):
    err = (pred - tgt) ** 2          # (B, 3)
    return (err * LOSS_WEIGHTS).mean()

@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss, total_ate = 0.0, 0.0
    n = 0
    for X, Y in loader:
        X, Y = X.to(DEVICE), Y.to(DEVICE)
        pred = model(X)
        total_loss += weighted_mse(pred, Y).item() * len(X)
        # ATE proxy: Euclidean position error (dx, dy only)
        ate = torch.sqrt(((pred[:,:2] - Y[:,:2])**2).sum(dim=1)).mean()
        total_ate += ate.item() * len(X)
        n += len(X)
    return total_loss / n, total_ate / n

print('✅ Training config ready')

In [ ]:
history = {'train_loss':[], 'val_loss':[], 'val_ate':[], 'lr':[]}
best_val = float('inf')
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for X, Y in tqdm(train_dl, desc=f'Epoch {epoch:02d}', leave=False):
        X, Y = X.to(DEVICE, non_blocking=True), Y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        pred = model(X)
        loss = weighted_mse(pred, Y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        train_loss += loss.item() * len(X)

    train_loss /= n_train
    val_loss, val_ate = evaluate(val_dl)
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_ate'].append(val_ate)
    history['lr'].append(current_lr)

    print(f'Epoch {epoch:02d} | train={train_loss:.5f} | val={val_loss:.5f} | ATE={val_ate:.3f}m | lr={current_lr:.2e}')

    if val_loss < best_val:
        best_val = val_loss
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_ate': val_ate,
            'scaler_mean': dataset.scaler.mean_.tolist(),
            'scaler_std':  dataset.scaler.scale_.tolist(),
            'config': {'d_model':128,'n_heads':12,'n_layers':4,'d_ff':512,'seq_len':SEQ_LEN},
        }, SAVE_PATH)
        print(f'  ✅ Saved best (val_loss={best_val:.5f})')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'  ⏹ Early stop at epoch {epoch}')
            break

print(f'\n🏁 Training complete. Best val_loss={best_val:.5f}')

## 7 · Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'],   label='Val')
axes[0].set(title='Loss (Weighted MSE)', xlabel='Epoch'); axes[0].legend()

axes[1].plot(history['val_ate'], color='tomato')
axes[1].set(title='Val ATE (m)', xlabel='Epoch', ylabel='metres')

axes[2].semilogy(history['lr'], color='grey')
axes[2].set(title='Learning Rate', xlabel='Epoch')

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
print('Saved training_curves.png')

## 8 · Test Set Evaluation

In [ ]:
# Load best checkpoint
ckpt = torch.load(SAVE_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
test_loss, test_ate = evaluate(test_dl)
print(f'Test loss : {test_loss:.5f}')
print(f'Test ATE  : {test_ate:.3f} m   ← target < 1.0 m')
print(f'Best epoch: {ckpt["epoch"]} | Val ATE: {ckpt["val_ate"]:.3f} m')

In [ ]:
# Visualise predicted vs ground-truth trajectory on a test sequence
model.eval()
X_vis, Y_vis = next(iter(test_dl))
with torch.no_grad():
    pred_vis = model(X_vis.to(DEVICE)).cpu().numpy()
Y_vis = Y_vis.numpy()

# Cumulative trajectory from per-window deltas
gt_traj  = np.cumsum(Y_vis[:200, :2], axis=0)
pred_traj= np.cumsum(pred_vis[:200, :2], axis=0)

plt.figure(figsize=(8, 6))
plt.plot(gt_traj[:,0],   gt_traj[:,1],   'b-',  label='Ground Truth', lw=2)
plt.plot(pred_traj[:,0], pred_traj[:,1], 'r--', label='DRIFTFormer',  lw=2)
plt.legend(); plt.title('Predicted vs GT Trajectory (first 200 windows)')
plt.xlabel('x (m)'); plt.ylabel('y (m)')
plt.axis('equal'); plt.grid(True, alpha=0.3)
plt.savefig('trajectory_comparison.png', dpi=150)
plt.show()

## 9 · Export Weights
Exports the model weights + scaler stats as JSON so the NAVDRIFT-0 dashboard can load them.

In [ ]:
import json

# Export full checkpoint
print('Checkpoint saved at:', SAVE_PATH)
print('Checkpoint keys:', list(ckpt.keys()))

# Export scaler stats for the dashboard (used to normalise live IMU in JS)
scaler_export = {
    'mean': ckpt['scaler_mean'],
    'std':  ckpt['scaler_std'],
    'channels': ['ax','ay','az','gx','gy','gz'],
    'seq_len': SEQ_LEN,
    'val_ate_m': round(ckpt['val_ate'], 4),
    'model_config': ckpt['config'],
}
with open('driftformer_scaler.json', 'w') as f:
    json.dump(scaler_export, f, indent=2)
print('Scaler exported → driftformer_scaler.json')
print(json.dumps(scaler_export, indent=2))

In [ ]:
# Export to ONNX for deployment (optional)
try:
    dummy = torch.randn(1, SEQ_LEN, 6).to(DEVICE)
    torch.onnx.export(
        model, dummy, 'driftformer.onnx',
        input_names=['imu_sequence'], output_names=['delta_xyt'],
        dynamic_axes={'imu_sequence':{0:'batch'}, 'delta_xyt':{0:'batch'}},
        opset_version=17,
    )
    print('✅ ONNX exported → driftformer.onnx')
except Exception as e:
    print('ONNX export skipped:', e)

In [ ]:
# Download all outputs to your local machine
from google.colab import files
files.download(SAVE_PATH)                  # PyTorch checkpoint
files.download('driftformer_scaler.json')  # Scaler stats for dashboard
files.download('training_curves.png')
files.download('trajectory_comparison.png')
try: files.download('driftformer.onnx')
except: pass

## 10 · Troubleshooting

| Issue | Fix |
|-------|-----|
| `No CSV files found` | Check repo structure — run `!find IO-VNBD -type f` |
| Column map wrong | Run the preview cell, copy exact column names into `COL` dict |
| OOM on A100 | Reduce `BATCH` from 512 → 256 |
| ATE > 5 m after 20 epochs | Check heading weight — increase `LOSS_WEIGHTS[2]` to 10.0 |
| Training loss NaN | Lower LR to `1e-4`, check for NaN in raw data (`df.isna().sum()`) |

---
**Target metrics for ISRO SIH 2026 demo:**  
- Val ATE < 1.0 m (60s outage window)  
- Heading RMSE < 2.0°  
- Inference latency < 5 ms on A100 (dashboard uses JS approx)
